In [ ]:
import numpy as np
import torch
import torch.nn as nn
from functions import train_flowmatching, euler_sampler, optimizing_learning_rate_seeds, validation_loss
from plot import plot_stacked_losses_errorbar, plot_samples_n_trajectories

import matplotlib.pyplot as plt

1. Optimizing learning rate

In [ ]:
#ranges = [(1e-5, 8e-1), (1e-4, 5e-1), (1e-3, 1e-1)]
ranges = [ (1e-3, 1e-1)] #(1e-5, 8e-1), (1e-4, 5e-1),
learn_rates = [np.geomspace(lo, hi, 9) for lo, hi in ranges]

n_train_steps = 2048
n_seeds = 5

means, stds = [], []
for lr in learn_rates:
    m, s = optimizing_learning_rate_seeds(lr, n_seeds=n_seeds, n_train_steps=n_train_steps)
    means.append(m)
    stds.append(s)

plot_stacked_losses_errorbar(learn_rates, means, stds,
                             titles=[f"{lo:.0e} to {hi:.0e}" for lo, hi in ranges],
                             n_train_steps=n_train_steps, n_seeds=n_seeds)

2. training final neural network and sampling

In [ ]:
#Daten erzeugen
mlp = nn.Sequential(nn.Linear(2, 64), nn.ReLU(),
                    nn.Linear(64, 64), nn.ReLU(),
                    nn.Linear(64, 1))
mlp, losses = train_flowmatching(mlp, 3e-2)

eval_gen = torch.Generator().manual_seed(349508)
x_normal = torch.randn(20000, 1, generator=eval_gen)
samples, traj = euler_sampler(mlp, x_normal)


In [ ]:
#plot_samples_n_trajectories(mlp, samples, traj, x_normal)
    

3. Visualizing